## Computational Aspects of Complex Reflection Groups

Götz Pfeiffer - University of Galway

# 3. Cosets: Enumerating Complex Reflection Groups

![Numbers](images/numbers.jpg)

## Setup

First, reload the algorithms from earlier ...

In [ ]:
using OrbitAl: Perm, PermGp, sizeOfGroup, transpositions, orbit, orbit_with_words
using OrbitAl: onPoints, onRight, onSets, onWords, plot_edges
using Graphs, GraphPlot
import Cairo, Fontconfig
using OrbitAl: presentations
using OrbitAl.variants: variantsRelations

## Complex Reflection Groups ...

... don't have
* a well-behaved length function
* simple reflections as generators
* root systems
* ...

But they do have
* a reflection representation
* parabolic subgroups
* ...
* a nice (Coxeter-like) presentation

Questions:
* how to construct the reflection representation systematically?
* how to turn the presentation into a usable group?

Here, we will only address the latter question, in the form of the **Todd-Coxeter coset enumeration** procedure,
a further variant of the orbit algorithm which potentially converts the group presentation into a finite permutation group. 

## Orbits with Data

* Recall all the modified orbit algorithms.
* Here is all in one:  words, transversal and images.

In [ ]:
function orbit_and_more(aaa, x, under)
    list = [x]
    index = Dict(x => 1)
    words = [[]]
    reps = [one(aaa[1])]
    images = [Int[] for a in aaa]
    for (i, y) in enumerate(list)
        for (k, a) in enumerate(aaa)
            z = under(y, a)
            l = get!(index, z) do
                push!(list, z)
                push!(words, onWords(words[i], k))
                push!(reps, onRight(reps[i], a))
                length(list)
            end
            push!(images[k], l)
        end
    end
    return (list = list, words = words, reps = reps, images = images)
end

In [ ]:
function orbit_and_data(aaa, x, under)
    data = (
        list = [x], 
        words = [[]], 
        reps = [one(aaa[1])], 
        images = [Int[] for a in aaa]
    )
    index = Dict(x => 1)
    for (i, y) in enumerate(data.list)
        for (k, a) in enumerate(aaa)
            z = under(y, a)
            l = get!(index, z) do
                push!(data.list, z)
                push!(data.words, onWords(data.words[i], k))
                push!(data.reps, onRight(data.reps[i], a))
                length(data.list)
            end
            push!(data.images[k], l)
        end
    end
    return data
end

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* The fields of a named tuple cannot be reassigned, but the lists they hold can still be modified, with `push!`.

</div>

* for example

In [ ]:
orb = orbit_and_data(transpositions(5), Set([1,2]), onSets)

In [ ]:
elist = union([(i, j) for img in orb.images for (i,j) in enumerate(img) if i != j])
plot_edges(elist)

* The image lists are the generators, acting as **permutations** of the positions $1, \dots, n$ of the orbit:

In [ ]:
[Perm(img) for img in orb.images]

* So the orbit is represented twice: by its elements, in `list`, and by their numbers $1, \dots, n$, on which the generators act by the permutations in `images`.
* The orbit algorithm needs the elements: `index` (or `z in list`) decides whether a new element $z$ has been seen before.
* Cosets $Hw$ of a subgroup $H$ of a group given by a presentation are different: whether $Hw = Hw'$ is exactly what is not known, and finding it out is the purpose of the enumeration.  So a coset cannot carry its own identity.  It can only be a **name**, here a number, whose equality with other names is decided later.
* This suggests an alternative design route: drop the elements, keep their numbers, their words, and the image lists, and record which numbers turn out to name the same coset.  There is a data structure for exactly that: **Union-Find**.

## Union-Find

* A coset enumeration will produce **coincidences**: pairs of numbers that turn out to name the same coset.
* Before we get there, let's look at the data structure that keeps track of such pairs: **Union-Find**.

<div class="alert alert-danger">

**Definition.**  Let $X$ be a set and let $R \subseteq X \times X$.  The **equivalence relation generated by** $R$ is the smallest equivalence relation $\sim$ on $X$ with $x \sim y$ for all $(x, y) \in R$.

</div>

* Union-Find does for equivalence relations what a stabilizer chain does for permutation groups:
  the structure is given by a list of **generators**, and the data structure answers **membership** questions.

| generators | data structure | question | answer |
|---|---|---|---|
| permutations $a \in A$ | stabilizer chain | $a \in \langle A \rangle$? | sift $a$ through the chain (`memberOfGroup`) |
| pairs $(x, y) \in R$ | union-find forest | $x \sim y$? | compare roots: `find(parent, x) == find(parent, y)` |
| vectors $v \in V$ | echelon basis | $v \in \langle V \rangle$? | reduce $v$ (Part 4) |

* In each case, a further generator can be added at any time.
* The classes of $\sim$ are also the orbits of the group generated by the transpositions $(x, y)$, $(x, y) \in R$.  So the orbit algorithm could find them too (see the Exercises).

### Find and Union

* Represent $X = \{1, \dots, n\}$ by a list `parent` of length $n$.  Each class of $\sim$ forms a **tree**, where `parent[x]` is the parent of $x$, and `parent[x] == x` if $x$ is the root.
* `find` follows the parent pointers from $x$ up to the root of its tree.
* `unite!` joins two trees by making one root the parent of the other: the larger root points to the smaller one, so that the root of each class is its **smallest** element.

In [ ]:
find(parent, x) = parent[x] == x ? x : find(parent, parent[x])

function unite!(parent, x, y)
    x, y = find(parent, x), find(parent, y)
    x == y && return false
    x > y && ((x, y) = (y, x))
    parent[y] = x    # the larger root points to the smaller one
    return true
end

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* By convention, the name of a function that **modifies** one of its arguments ends in `!`, like `push!` and `get!`.
* `(x, y) = (y, x)` swaps the values of `x` and `y`.

</div>

* For example, on $n = 10$ points:

In [ ]:
n = 10
pairs = [(1, 6), (4, 9), (6, 2), (8, 3), (9, 10), (2, 7), (3, 5)]
parent = collect(1:n)
for (x, y) in pairs
    unite!(parent, x, y)
end
parent

* The classes, each listed under its root, the smallest element of the class (just like the `Orbit` objects in Part 1, which compare by their first element):

In [ ]:
classes = [[x for x in 1:n if find(parent, x) == r] for r in 1:n if parent[r] == r]

In [ ]:
find(parent, 3) == find(parent, 7), find(parent, 3) == find(parent, 5)

* And the forest itself:

In [ ]:
plot_edges([(x, parent[x]) for x in 1:n if parent[x] != x])

* Union-Find has a linear version, where the pairs are replaced by vectors.  We'll meet it in Part 4.

## Example

* The complex reflection group $G_{12}$ has a presentation 
$$
\langle
s_1, s_2, s_3 \mid
s_1^2 = s_2^2 = s_3^2 = 1,\,
s_1 s_2 s_3 s_1 = s_2 s_3 s_1 s_2 = s_3 s_1 s_2 s_3
\rangle
$$
* Let's try and enumerate its elements systematically.

## Coset Enumeration

* Q: What is $G = \langle S \mid R \rangle$?
* A: Todd-Coxeter!

* Suppose that a group $G$ is given by a **presentation** $\langle S \mid R \rangle$, consisting of a (finite) set $S$ of abstract **generators** $s_1, s_2, \dots, s_k$, and a (finite) list $R$ of **relations** $l_j = r_j$, where both $l_j$ and $r_j$ are words in $S \cup S^{-1}$.

* For convenience, we assume that $S$ is closed under inverses: $S  = S^{-1}$.

* We wish to enumerate the elements of $G$ (hoping that $G$ is a finite group), or more generally, the cosets of a subgroup $H$ of $G$ (hoping that $H$ has finite index in $G$).

* A priori, neither the domain $X$ being acted upon (by $G$), nor the edges of the action graph are known.

* Strategy: define new cosets as images of old cosets under a generator, but be prepared to identify a new coset with an existing one, if the relations imply they are the same.

* Eventually, we want $x.s$ to be known for all cosets $x$ and all $s \in S$.

### The Coset Table

* A presentation `genrel` of $G$ has components
  * `gens`: a list `1:k` of abstract generators $S = S^{-1}$
  * `invr`: the map $S \to S: s \mapsto s^{-1}$, as a list
  * `rels`: a list of relations, as pairs of words in $S$
  * `sbgp`: a list of words in $S$, generating a subgroup $H$ of $G$.
* A **coset table** lists, for each coset $x$ found so far and each generator $s$, the coset $x.s$, or $0$ if it is not known yet.
* The cosets are numbers $1, 2, 3, \dots$, and each has a word in $S$, the path by which it was found, as in `orbit_and_data`.
* Cosets that turn out to be equal are merged, in a Union-Find forest on the cosets: `parent[x] == x` if $x$ is **active**, and otherwise `parent[x]` points towards the active coset that $x$ has been merged into.
* Pairs of cosets that are found to be equal wait in a `queue`, until they are merged (see *Coincidences* below).

In [ ]:
mutable struct CosetTable
    next::Vector{Vector{Int}}             # next[x][s], 0 = not yet known
    parent::Vector{Int}                   # the Union-Find forest on the cosets
    queue::Vector{Tuple{Int,Int}}         # the pending coincidences
    word::Vector{Vector{Int}}             # the word that defined each coset
    invr::Vector{Int}                     # invr[s] is the inverse of s
    variants::Vector{Vector{Vector{Int}}} # the relation variants, see below
    active::Int                           # the number of active cosets
end

CosetTable(genrel) = CosetTable([], [], [], [], genrel.invr, variantsRelations(genrel), 0)

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* A `mutable struct` is a `struct` whose fields can be reassigned after construction, like `T.active` below ([mutable composite types](https://docs.julialang.org/en/v1/manual/types/#Mutable-Composite-Types)).  The lists it holds can be modified with `push!` in either case.
* `Vector{Vector{Int}}` is a list of lists of integers: `next[x]` is the row of coset `x`, and `next[x][s]` its entry for generator `s`.

</div>

* `find` follows the parent pointers to the active coset, as in the Union-Find section, and `image` looks up $x.s$, as an active coset, or $0$.

In [ ]:
function find(T::CosetTable, x)
    while T.parent[x] != x
        x = T.parent[x]
    end
    return x
end

is_active(T, x) = T.parent[x] == x

image(T, x, s) = (y = T.next[x][s]; y == 0 ? 0 : find(T, y))

* A new coset $y = x.s$ gets a new number, an empty row, and the word of $x$ followed by $s$.  Its entry for $s^{-1}$ is $x$.

In [ ]:
function newcoset!(T, word)
    push!(T.next, zeros(Int, length(T.invr)))
    push!(T.parent, length(T.parent) + 1)
    push!(T.word, word)
    T.active += 1
    return length(T.parent)
end

function sprout!(T, x, s)
    y = newcoset!(T, [T.word[x]; s])
    T.next[x][s] = y
    T.next[y][T.invr[s]] = x
    return y
end

* For display, a coset table lists its active cosets, with the word that defined them, and their images.

In [ ]:
function Base.show(io::IO, ::MIME"text/plain", T::CosetTable)
    acti = filter(x -> is_active(T, x), eachindex(T.parent))
    println(io, "CosetTable: ", length(acti), " active of ", length(T.parent), " cosets")
    for x in acti[1:min(end, 12)]
        println(io, lpad(x, 4), ": ", rpad(join(T.word[x]), 10), [image(T, x, s) for s in eachindex(T.invr)])
    end
    length(acti) > 12 && print(io, "   ...")
end

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* `show(io, x)` determines how `x` is printed ([custom printing](https://docs.julialang.org/en/v1/manual/types/#man-custom-pretty-printing)).  Defining `Base.show` with its full name adds a method to it without `import Base: show` first.
* The variant `show(io, ::MIME"text/plain", x)` is used when an object is displayed on its own, for example as the result of a cell; the shorter `show(io, x)` when it appears inside another object, such as a list.

</div>

### Actions on Words

* There are two ways of applying a word to a coset: a **partial** action, which gives up (returns $0$) when it meets an unknown image, and a **defining** action, which defines new cosets when necessary.

In [ ]:
function underWordPartial(T, x, word)
    for s in word
        x = image(T, x, s)
        x == 0 && return 0
    end
    return x
end

function underWordSprout!(T, x, word)
    for s in word
        y = image(T, x, s)
        x = y == 0 ? sprout!(T, x, s) : y
    end
    return x
end

### Coincidences

* In the action graph, an edge $x \stackrel{s}{\longrightarrow} y$ always comes with the opposite edge $y \stackrel{s^{-1}}{\longrightarrow} x$.  So to set $x.s = y$ means to set two entries of the table: $x.s = y$ and $y.s^{-1} = x$: `link!` calls `setImage!` twice.
* If an entry is already set, to a different coset, then the two cosets are equal: a **coincidence**.
* Merging two cosets means: make the larger one point to the smaller one, as `unite!` does in the Union-Find section, and move all entries of its row over to the smaller one, which can cause further coincidences.  So the coset table gets its own method of `unite!`: Union-Find **with consequences**.
* The pending coincidences are kept in the queue `T.queue`.  The loop over the queue grows while it runs, just like the list in the orbit algorithm.  When it is done, the queue is emptied, ready for the next edge.

<div class="alert alert-info">

**Coincidence Procedure**
* **Input:** a coset table, and a new edge $x \stackrel{s}{\longrightarrow} y$.
* **1.** Set $x.s = y$ and $y.s^{-1} = x$.  Each clash with an existing entry is a pair of equal cosets: put it in the **queue**.
* **2.** For each pair $(a, b)$ in the queue: replace $a$ and $b$ by their active cosets.  If they are different, with $a < b$, then **unite** them: $b$ now points to $a$, and each entry $b.t$ is moved over to $a$ as in step 1, possibly adding further pairs to the queue.
* **3.** When the queue is exhausted, the table is consistent again.

</div>

In [ ]:
# x.s = y, or a coincidence if x.s is already set
function setImage!(T, x, s, y)
    x, y = find(T, x), find(T, y)
    z = T.next[x][s]
    z == 0 ? (T.next[x][s] = y) : push!(T.queue, (y, z))
end

# x.s = y and y.s^-1 = x
function link!(T, x, s, y)
    setImage!(T, x, s, y)
    setImage!(T, y, T.invr[s], x)
end

# merge the cosets a and b, and their rows
function unite!(T::CosetTable, a, b)
    a, b = find(T, a), find(T, b)
    a == b && return false
    a > b && ((a, b) = (b, a))
    T.parent[b] = a                        # the larger coset points to the smaller one
    T.active -= 1
    for (t, y) in enumerate(T.next[b])     # move b's edges over to a
        y == 0 || link!(T, a, t, y)
    end
    return true
end

function updateEdge!(T, x, s, y)
    link!(T, x, s, y)
    for (a, b) in T.queue                      # the queue grows while we loop
        unite!(T, a, b)
    end
    empty!(T.queue)                            # ready for the next edge
end

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* The type annotation `T::CosetTable` makes `unite!(T, a, b)` a new **method** of `unite!`, used only for coset tables.  Without it, the definition would replace `unite!(parent, x, y)` from the Union-Find section, which takes three arguments, too.
* `for (a, b) in T.queue` is safe while `push!` adds to `T.queue`: like the list of the orbit algorithm, the loop sees the new entries.
* `empty!(T.queue)` removes all entries of the list, but keeps the list object (and the memory it has allocated) for reuse.

</div>

### Relations

* A relation $l = r$ is a relator $w = l r^{-1}$, a word that should act trivially on every coset.
* If $w = u s v$ for a generator $s$, then $s = u^{-1} v^{-1}$, and so $x.s = x.(v u)^{-1}$ for each coset $x$.  These words $(vu)^{-1}$ are the **variants** of the relations for $s$, computed by `variantsRelations`.
* For example, here are the relations of $G_{12}$, and the variants for its first generator:

In [ ]:
G = presentations.G12
G.rels, variantsRelations(G)[1]

* To **trace** a word $w$ from coset $x$ means to make sure that $x.w = x$, defining new cosets as necessary.  If $w \in H$ then $x.w = x$ should hold for the trivial coset $x = H$, which is coset $1$.  So the words generating $H$ are traced from coset $1$.
* To **finalize** coset $x$ under generator $s$: try each variant $w$ for $s$, and if $x.w$ is already known, set $x.s = x.w$, possibly causing coincidences.  If $x.s$ is still unknown after that, define it as a new coset.

In [ ]:
function trace!(T, x, word)
    y = underWordSprout!(T, x, word[1:end-1])
    updateEdge!(T, y, word[end], x)
end

function finalize!(T, x, s)
    for variant in T.variants[s]
        is_active(T, x) || return
        y = underWordPartial(T, x, variant)
        y == 0 || updateEdge!(T, x, s, y)
    end
    is_active(T, x) && T.next[x][s] == 0 && sprout!(T, x, s)
end

### Enumerate!

* The enumeration is an orbit algorithm over the coset numbers: start with coset $1$ and the subgroup tables, then finalize each coset in turn under each generator, while new cosets are being defined.

<div class="alert alert-info">

**Coset Enumeration**
* **Input:** a presentation $\langle S \mid R \rangle$ of a group $G$, and words generating a subgroup $H$.
* **Output:** the coset table of $H$ in $G$, if $|G : H|$ is finite.
* **1. Initialize** the table with coset $1$, and trace the generators of $H$ from coset $1$.
* **2. Loop** over the cosets $x = 1, 2, \dots$, as long as there are more, and finalize $x$ under each $s \in S$.
* **3. Return** the table.  Its active cosets are the cosets of $H$ in $G$.

</div>

In [ ]:
function coset_table(genrel, sbgp)
    T = CosetTable(genrel)
    x = newcoset!(T, Int[])
    for word in sbgp
        trace!(T, x, word)
    end
    x = 0
    while x < length(T.parent)                # the table grows while we loop
        x += 1
        for s in genrel.gens
            finalize!(T, x, s)
        end
    end
    return T
end

* A loop `for x in 1:length(T.parent)` would not do here: its range is fixed when the loop starts.  Hence the `while` loop.
* Inactive cosets are visited too, but `finalize!` returns immediately for them.

## Test Drive

* That's it.  Let's try $G_{12}$, over the subgroup generated by its first generator:

In [ ]:
T = coset_table(G, G.sbgp)

* How to convert the table into a permutation group: renumber the active cosets $1, \dots, n$, and turn the image lists into permutations.

In [ ]:
function perms(T)
    acti = filter(x -> is_active(T, x), eachindex(T.parent))
    idx = Dict(x => i for (i, x) in enumerate(acti))
    [Perm([idx[image(T, x, s)] for x in acti]) for s in eachindex(T.invr)]
end

import OrbitAl: PermGp
PermGp(T::CosetTable) = (gens = perms(T); PermGp(gens, one(gens[1])))

In [ ]:
gp = PermGp(T)
sizeOfGroup(gp)

* Finally: plot the action graph!

In [ ]:
elist = union([(i, j) for a in gp.gens for (i, j) in enumerate(a.list) if i != j])
plot_edges(elist)

* $G(3,3,3)$, over the subgroup $\langle t_0, t_1, t_2 \rangle$, as in Part 4:

In [ ]:
G = presentations.G333
T = coset_table(G, G.sbgp)

* The same nine cosets appear in the hand-made coset table of the Hecke algebra of $G(3,3,3)$ in Part 4 (with generator 4 as $s_3$, and generators $1, 2, 3$ as $t_0, t_1, t_2$).  The words of $x_1, \dots, x_4$ agree exactly: $s_3$, $s_3 t_0$, $s_3 t_1$, $s_3 t_2$.  The later ones differ, since the hand-made table defines its cosets in a different order.

In [ ]:
sizeOfGroup(PermGp(T))

* For larger enumerations, take the trivial subgroup: the cosets are then the elements of the group.

In [ ]:
for name in [:G29, :M12, :J1]
    G = getfield(presentations, name)
    t = @elapsed T = coset_table(G, Vector{Int}[])
    println(rpad(name, 5), "order ", lpad(T.active, 6), ", cosets defined ", lpad(length(T.parent), 6), ", ", round(t, digits = 2), "s")
end

## Cosets as Objects

* The motto of the notebooks is: if you can define a group action on your kind of objects, then all the orbit machinery applies.  Here, the cosets have become mere numbers.  But a thin **handle** gives them back their behaviour: a `Coset` is a reference to a table, together with a coset number.
* The data stays in the table.  A `Coset` knows where to find it: its word, its images, and which active coset it stands for.

In [ ]:
struct Coset
    table::CosetTable
    id::Int
end

active(c::Coset) = find(c.table, c.id)          # the active coset that c stands for
word(c::Coset) = c.table.word[active(c)]

function Base.:^(c::Coset, s::Int)              # the coset c.s, or nothing if unknown
    y = image(c.table, active(c), s)
    y == 0 ? nothing : Coset(c.table, y)
end

Base.:(==)(c::Coset, d::Coset) = c.table === d.table && active(c) == active(d)
Base.hash(c::Coset, h::UInt) = hash(active(c), h)
Base.isless(c::Coset, d::Coset) = active(c) < active(d)
Base.show(io::IO, c::Coset) = print(io, "Coset(", active(c), ", word = ", join(word(c)), ")")

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* `Base.:^` and `Base.:(==)` are the full names of the operators `^` and `==`: defining them adds methods for `Coset`s.
* Objects that are `==` must have the same `hash`, since `Set` and `Dict` rely on it.  So whenever `==` gets a new method, `hash` needs a matching one ([hash](https://docs.julialang.org/en/v1/base/base/#Base.hash)).
* `c.table === d.table` tests whether two cosets refer to the very same table object, not just to equal ones.

</div>

* Two cosets are equal if they stand for the same active coset, so they are compared, and hashed, by `active(c)`: Union-Find provides the equality test that cosets lack.
* For example, in $G(3,3,3)$:

In [ ]:
G = presentations.G333
T = coset_table(G, G.sbgp)
x = Coset(T, 1)
x^4, (x^4)^1, ((x^4)^1)^4

* With `onCosets` as action function, the orbit algorithm of Part 1 applies to cosets, like to any other objects.  The orbit of the trivial coset under the generators is the set of all cosets:

In [ ]:
onCosets(c, s) = c^s
orb = orbit(G.gens, x, onCosets)
length(orb) == T.active

* And `orbit_with_words` gives each coset a word, that is, a **Schreier transversal** of the subgroup, as needed for the Reidemeister-Schreier method:

In [ ]:
orbit_with_words(G.gens, x, onCosets).words

* So the cosets do form an orbit, in exactly the sense of Part 1.  What the enumeration adds is the step *before* that: deciding which cosets are equal.

## Remarks

* **Termination.**  Coset enumeration terminates if the index $|G : H|$ is finite, provided that the strategy is fair: every entry of the table is eventually defined or deduced, and every relator is eventually checked at every coset.  But there is no bound on the number of cosets defined along the way in terms of the index, and whether a finite presentation defines a finite group cannot be decided in general [Neu82], [Sim94], [HEO05].
* **Strategies.**  The strategy here, deducing from all variants of the relations before each definition, is close to the Felsch strategy.  The Haselgrove-Leech-Trotter (HLT) strategy instead defines cosets freely while tracing relators, and typically defines many more cosets, but is cheaper per coset [Sim94], [HEO05].

##  Exercises, etc.

**Exercise 3.1.**  ($*$) Modify the coset enumeration procedure so that it applies to monoid presentations.

**Exercise 3.2.**  Find matrices for the reflection representation of a given complex reflection group $G$.

**Exercise 3.3.**  Find a way to enumerate the (conjugacy classes of) parabolic subgroups $P$ of a complex reflection group $G$.

**Exercise 3.4.**  Compute the normalizer of parabolic subgroup $P$ in $G$.  Does $P$ always have a complement $H$? If so, does $H$ have a natural set of generators? 

**Exercise 3.5** (Path Compression).  Modify `find` into a function `find!(parent, x)` that also makes each node on the path from $x$ to its root point directly at the root.  Why is this allowed?  What does it achieve?

In [ ]:
function find!(parent, x)
    # your code here: return the root of x, and point each node on the way directly at it
end

In [ ]:
n = 1000
pairs = [(rand(1:n), rand(1:n)) for _ in 1:600]
parent = collect(1:n)
for (x, y) in pairs
    unite!(parent, x, y)
end
function check_find!()
    isnothing(find!(collect(1:2), 1)) && return false
    p = collect(1:n)
    for (x, y) in pairs
        x, y = find!(p, x), find!(p, y)
        x == y && continue
        x > y && ((x, y) = (y, x))
        p[y] = x
    end
    roots = [find!(p, x) for x in 1:n]
    roots == [find(parent, x) for x in 1:n] && all(p[x] == roots[x] for x in 1:n)
end
check_find!()

**Exercise 3.6** (BFS vs. Union-Find).  Show that the classes of the equivalence relation generated by $R \subseteq X \times X$ are the orbits of the group generated by the transpositions $(x, y)$, $(x, y) \in R$, $x \neq y$.  Write a function `classes_by_orbits(n, pairs)` that computes these classes with the orbit algorithm.  Compare it with union-find: which approach is preferable if the pairs are all known in advance (offline), and which if they arrive one at a time (online), as the coincidences in a coset enumeration do?

In [ ]:
function classes_by_orbits(n, pairs)
    # your code here: return the list of classes, each as a list of points
end

In [ ]:
n = 12
pairs = [(2, 5), (7, 3), (5, 9), (11, 12), (3, 10), (9, 2)]
parent = collect(1:n)
for (x, y) in pairs
    unite!(parent, x, y)
end
theirs = Set(Set(x for x in 1:n if find(parent, x) == r) for r in 1:n if parent[r] == r)
mine = classes_by_orbits(n, pairs)
!isnothing(mine) && Set(Set.(mine)) == theirs

**Exercise 3.7.**  Use `find!` from Exercise 3.5 in the coset enumeration, and measure the effect on the enumerations over the trivial subgroup.

**Exercise 3.8.**  Process the coincidence queue last in, first out, as a stack, instead of first in, first out.  Does the result change?  The number of steps?

**Exercise 3.9.**  Write `compact!(T)`, which renumbers the active cosets $1, \dots, n$ and discards the others, during the enumeration, when more than half of the cosets have become inactive.

**Exercise 3.10.**  Implement the HLT strategy: for each coset $x$ and each relator $r$, trace $r$ from $x$.  Compare the number of cosets defined with the strategy above, on $M_{12}$ and $J_1$.

**Exercise 3.11.**  ($*$) Check whether the strategy above is fair, in the sense of the remark on termination.

* ...